# Cross-sectional Oracle/HITS vs congressional-event HITS

Run and inspect the canonical cross-sectional Oracle and HITS models, plus the only added variant: HITS labels built from congressional event-date nodes and edges. Every model is trained on historical data and scored/backtested across every daily date in each forward walk-forward year.

The experiment is cache-aware: set `RUN_EXPERIMENT = True` to execute the script, or leave it `False` to inspect an existing artifact directory.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

import pandas as pd
import matplotlib.pyplot as plt

# Make the notebook work when launched from either the repository root or notebooks/.
HERE = Path.cwd().resolve()
REPO = HERE.parent if HERE.name == "notebooks" else HERE
SCRIPT = REPO / "scripts" / "run_congress_cross_rank_comparison.py"
OUT = REPO / "artifacts" / "congress_cross_rank_comparison"

RUN_EXPERIMENT = False
TIERS = "1T,100B,10B"  # configured tiers
FIRST_YEAR = 2021
LAST_YEAR = 2025
TOP_K = 10

OUT.mkdir(parents=True, exist_ok=True)
if RUN_EXPERIMENT:
    env = os.environ.copy()
    env.update({
        "CONGRESS_COMPARE_OUT": str(OUT),
        "CROSS_RANK_TIERS": TIERS,
        "CROSS_RANK_FIRST_YEAR": str(FIRST_YEAR),
        "CROSS_RANK_LAST_YEAR": str(LAST_YEAR),
        "CROSS_RANK_TOP_K": str(TOP_K),
    })
    subprocess.run([sys.executable, str(SCRIPT), "--universe", TIERS], cwd=REPO, env=env, check=True)

print({"repo": str(REPO), "output": str(OUT), "run_experiment": RUN_EXPERIMENT})

In [ ]:
results_path = OUT / "all_results.csv"
if not results_path.exists():
    raise FileNotFoundError(f"No results at {results_path}. Set RUN_EXPERIMENT=True first.")

results = pd.read_csv(results_path, parse_dates=["train_start", "train_end"])
results["model_variant"] = results["model"].astype(str) + " / " + results["variant"].astype(str)

summary = (
    results.groupby(["tier", "year", "top_k", "model", "variant", "ranker"], as_index=False)
    .agg(
        families=("family", "nunique"),
        mean_return=("total_return", "mean"),
        median_return=("total_return", "median"),
        mean_sharpe=("sharpe", "mean"),
        total_trades=("trades", "sum"),
    )
)
selected_tier = TIERS.split(",")[0].strip().upper()
display(summary.query("tier == @selected_tier and top_k == @TOP_K").sort_values(["year", "model", "ranker", "variant"]))

In [ ]:
# Side-by-side yearly comparison of the three score sets.
tier = TIERS.split(",")[0].strip().upper()
view = summary.loc[(summary.tier == tier) & (summary.top_k == TOP_K)].copy()
view["label"] = view["model"] + " / " + view["ranker"] + " / " + view["variant"]
returns = view.pivot(index="year", columns="label", values="mean_return")
ax = returns.plot(marker="o", figsize=(12, 5), title=f"Congress graph vs regular HITS vs Oracle — {tier}, top-k={TOP_K}")
ax.set_ylabel("Mean total return across feature families")
ax.set_xlabel("Test year")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Aggregate view: the most useful quick read for model selection.
display(
    view.groupby("label", as_index=False)
    .agg(mean_return=("mean_return", "mean"), mean_sharpe=("mean_sharpe", "mean"), total_trades=("total_trades", "sum"))
    .sort_values("mean_return", ascending=False)
)